# Document Classification Example

Evaluates LLMs on 16-class document classification using [RVL-CDIP](https://huggingface.co/datasets/dvgodoy/rvl_cdip_mini) (OCR-extracted text).

Uses a **QuestionPipeline**: **TemplateQuestionGenerator** → **QuestionRenderer** → **RolloutGenerator**, then scores with `compute_metrics_summary()`.

In [ ]:
%pip install lightningrod-ai python-dotenv datasets pandas

from IPython.display import clear_output
clear_output()

## Set up the client

Sign up at [dashboard.lightningrod.ai](https://dashboard.lightningrod.ai/?redirect=/api) to get your API key and **$50 of free credits**.

- **Google Colab**: Go to the Secrets section (key icon in left sidebar) and add a secret named `LIGHTNINGROD_API_KEY`
- **Local Jupyter**: Set the `LIGHTNINGROD_API_KEY` environment variable, or you'll be prompted to enter it

In [ ]:
from dotenv import load_dotenv
from lightningrod import LightningRod
from lightningrod.utils import config

load_dotenv()

api_key = config.get_config_value("LIGHTNINGROD_API_KEY")
lr = LightningRod(api_key=api_key)


## Download and prepare benchmark data

We use the [dvgodoy/rvl_cdip_mini](https://huggingface.co/datasets/dvgodoy/rvl_cdip_mini) dataset from HuggingFace — a 4,000-document subset of the RVL-CDIP benchmark with 16 document types and pre-extracted OCR text.

Each document is converted to a `Sample` with the OCR text as the seed and the ground-truth label stored in metadata for later evaluation.

In [8]:
import random
from datasets import load_dataset
from lightningrod import create_sample
from typing import List, Tuple

LABEL_NAMES = [
    "letter", "form", "email", "handwritten", "advertisement",
    "scientific_report", "scientific_publication", "specification",
    "file_folder", "news_article", "budget", "invoice",
    "presentation", "questionnaire", "resume", "memo",
]

MIN_TEXT_LENGTH = 50
MAX_DOC_LENGTH = 4000
NUM_SAMPLES = 20

ds = load_dataset("dvgodoy/rvl_cdip_mini", split="test")

# Keep only non-image columns
if "image" in ds.column_names:
    ds = ds.remove_columns(["image"])
    
text_and_labels: List[Tuple[str, str]] = []
for ex in ds:
    paragraphs = ex.get("ocr_paragraphs") or []
    doc_text = "\n\n".join(paragraphs).strip()
    if len(doc_text) < MIN_TEXT_LENGTH:
        continue
    if len(doc_text) > MAX_DOC_LENGTH:
        doc_text = doc_text[:MAX_DOC_LENGTH] + "\n... (truncated)"

    label_name = LABEL_NAMES[ex["label"]]
    text_and_labels.append((doc_text, label_name))

# Construct Samples:
samples = []
for (text, label) in text_and_labels:
    samples.append(create_sample(seed_text=text, label=label))

random.seed(42)
samples = random.sample(samples, min(NUM_SAMPLES, len(samples)))
print(f"{len(samples)} samples ready for evaluation")

20 samples ready for evaluation


## Upload input dataset

In [9]:
input_dataset = lr.datasets.create_from_samples(samples)
print(f"Created input dataset: {input_dataset.id}")
print(f"Total samples: {input_dataset.num_rows}")

Created input dataset: 7392e1c8-e5df-42fe-92e2-2bd2c3c1446c
Total samples: 20


## Configure the pipeline

The pipeline has three stages:
1. **TemplateQuestionGenerator** — fills a classification prompt template with each document's OCR text
2. **QuestionRenderer** — renders the question with a multiple-choice answer type
3. **RolloutGenerator** — sends the rendered prompt to multiple LLMs via OpenRouter

In [14]:
from lightningrod import (
    QuestionPipeline,
    TemplateQuestionGenerator,
    QuestionRenderer,
    RolloutGenerator,
    MultipleChoiceAnswerType,
    ModelConfig,
    ModelSourceType,
    RolloutScorer,
)

OPTIONS = {f"option_{i}": name for i, name in enumerate(LABEL_NAMES)}

_options_display = "\n".join(f"{chr(65 + i)}) {name}" for i, name in enumerate(LABEL_NAMES))
QUESTION_TEMPLATE = (
    "What type of document is this? Classify it as one of the following categories:\n\n"
    f"{_options_display}\n\n"
    "Document text:\n{seed_text}"
)

models = [
    ModelConfig(model_name="openai/gpt-5.2", model_source=ModelSourceType.OPEN_ROUTER, use_pipeline_key=True),
    ModelConfig(model_name="anthropic/claude-sonnet-4.6", model_source=ModelSourceType.OPEN_ROUTER, use_pipeline_key=True),
    ModelConfig(model_name="google/gemini-3.1-pro-preview", model_source=ModelSourceType.OPEN_ROUTER, use_pipeline_key=True),
]

answer_type = MultipleChoiceAnswerType()

pipeline = QuestionPipeline(
    question_generator=TemplateQuestionGenerator(question_template=QUESTION_TEMPLATE),
    renderer=QuestionRenderer(answer_type=answer_type),
    rollout_generator=RolloutGenerator(models=models),
    scorer=RolloutScorer(answer_type=answer_type, multiple_choice_options=OPTIONS),
)

## Run the pipeline

This sends each document to all three models for classification. It may take a few minutes depending on the number of samples.

In [15]:
dataset = lr.transforms.run(
    pipeline,
    input_dataset=input_dataset,
    name="Document Classification",
)

Output()

## View results

Download the results and compute per-model accuracy using `compute_metrics_summary()`.

In [16]:
import pandas as pd
from lightningrod.utils import compute_metrics_summary

result_samples = dataset.download()

summary = compute_metrics_summary(result_samples, OPTIONS)
df = pd.DataFrame.from_dict(summary, orient="index")
df.index.name = "model"
df

,accuracy,n_correct,n_parsed,n_total,parse_rate,mean_reward
model,,,,,,
openai/gpt-5.2,0.526316,10,19,20,0.95,-1.915430
anthropic/claude-sonnet-4.6,0.550000,11,20,20,1.00,-1.413900
google/gemini-3.1-pro-preview,0.650000,13,20,20,1.00,-1.661844


In [ ]:
from lightningrod.utils import compute_multi_choice_consensus, multi_choice_log_score

consensus_rows = compute_multi_choice_consensus(result_samples, OPTIONS)
labeled = [r for r in consensus_rows if r["label"] is not None]

n_agree = sum(1 for r in consensus_rows if r["all_agree"])
print(f"Samples with 2+ parsed rollouts: {len(consensus_rows)}")
print(f"All models agree: {n_agree}/{len(consensus_rows)} ({n_agree/len(consensus_rows):.0%})\n")

print("Consensus vs individual model accuracy (over samples with 2+ rollouts):")
rows = {"consensus": {
    "accuracy": sum(r["consensus_answer"] == r["label"] for r in labeled) / len(labeled),
    "mean_log_score": sum(multi_choice_log_score(r["consensus"], r["label"], OPTIONS) for r in labeled) / len(labeled),
    "n": len(labeled),
}}
for model in sorted({m for r in labeled for m in r["predictions"]}):
    ml = [r for r in labeled if model in r["per_model_answers"]]
    rows[model] = {
        "accuracy": sum(r["per_model_answers"][model] == r["label"] for r in ml) / len(ml),
        "mean_log_score": sum(multi_choice_log_score(r["predictions"][model], r["label"], OPTIONS) for r in ml if model in r["predictions"]) / len(ml),
        "n": len(ml),
    }
display(pd.DataFrame.from_dict(rows, orient="index").rename_axis("model"))

print("Agreement split (does unanimous agreement predict higher accuracy?):")
agree = [r for r in labeled if r["all_agree"]]
disagree = [r for r in labeled if not r["all_agree"]]
def _stats(subset):
    if not subset:
        return {"n": 0, "consensus_accuracy": None, "consensus_mean_log_score": None}
    return {
        "n": len(subset),
        "consensus_accuracy": sum(r["consensus_answer"] == r["label"] for r in subset) / len(subset),
        "consensus_mean_log_score": sum(multi_choice_log_score(r["consensus"], r["label"], OPTIONS) for r in subset) / len(subset),
    }
display(pd.DataFrame({"all": _stats(labeled), "agreement_set": _stats(agree), "disagreement_set": _stats(disagree)}).T.rename_axis("subset"))

Samples with 2+ parsed rollouts: 20
All models agree: 13/20 (65%)

Consensus vs individual model accuracy (over samples with 2+ rollouts):


,accuracy,mean_log_score,n
model,,,
consensus,0.700000,-1.187464,20
anthropic/claude-sonnet-4.6,0.550000,-1.413900,20
google/gemini-3.1-pro-preview,0.650000,-1.661844,20
openai/gpt-5.2,0.526316,-1.595189,19


Agreement split (does unanimous agreement predict higher accuracy?):


,n,consensus_accuracy,consensus_mean_log_score
subset,,,
all,20.0,0.700000,-1.187464
agreement_set,13.0,0.692308,-0.981030
disagreement_set,7.0,0.714286,-1.570841
